In [0]:
from pyspark.sql import functions as F

catalog = "workspace"
schema_name = "default"

silver_table = f"`{catalog}`.`{schema_name}`.`orders_silver`"
gold_table = f"`{catalog}`.`{schema_name}`.`customer_revenue_gold`"

silver_df = spark.table(silver_table)

gold_df = (
    silver_df
    .filter(F.col("status") == "Delivered")
    .groupBy("customer")
    .agg(
        F.sum("amount").alias("total_revenue"),
        F.count("order_id").alias("total_orders"),
        F.round(F.avg("amount"), 2).alias("average_order_value")
    )
)

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_table)

total_revenue = spark.table(gold_table).agg(
    F.sum("total_revenue")
).first()[0]

assert total_revenue == 66000.0

print("Gold completed successfully")